## Load Required Libraries

This notebook builds a machine learning model to predict NCAA tournament seeds using historical data.

The first step is importing the Python libraries required for data processing, modeling, and evaluation. These libraries provide tools for:

- Reading and manipulating datasets
- Performing numerical transformations
- Training machine learning models
- Evaluating model performance

The libraries imported include:

- **pandas** – Used for loading and manipulating tabular datasets (CSV/Excel files).
- **numpy** – Provides numerical operations and mathematical functions.
- **scikit-learn metrics** – Used to evaluate machine learning model performance.
- **train_test_split / cross_val_score** – Used for training/testing splits and cross-validation.
- **GradientBoostingRegressor** – A regression model used to predict NCAA tournament seeds.

These tools form the foundation for the modeling workflow used throughout the notebook.

In [ ]:
import pandas as pd
import numpy as np
from sklearn import metrics
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import GroupKFold


## Load Data

This section loads the datasets used to train and evaluate the model.

The datasets include:

- **Historical NCAA tournament training data**
- **Conference ranking data stored across multiple Excel sheets**

The training dataset contains historical team statistics and tournament outcomes. These records are used to train the model so it can learn relationships between team performance metrics and tournament seed placement.

Conference rankings are stored in separate sheets for each season, which allows the model to retrieve the correct ranking based on the season being processed.

In [ ]:
training = pd.DataFrame()
training = pd.read_excel("NCAA_Seed_Training_Set2.0.xlsx", sheet_name="NCAA_Seed_Training_Set2.0")
training_sheets = pd.read_excel("NCAA_Seed_Training_Set2.0.xlsx", sheet_name=None)
testing = pd.DataFrame()
testing = pd.read_excel("NCAA_Seed_Test_Set2.0.xlsx", sheet_name="NCAA_Seed_Test_Set2.0")
testing_sheets = pd.read_excel("NCAA_Seed_Training_Set2.0.xlsx", sheet_name=None)

## Retrieve Conference Rank for Each Team

This cell defines a function that retrieves the conference ranking for each team in the training dataset.

Steps performed:

1. **Define Function**
   - `get_conf_rank()` takes a row of data and the dictionary of conference ranking sheets.

2. **Identify the Season**
   - The function extracts the `Season` value from the row.

3. **Identify the Conference**
   - The `Conference` column indicates which conference the team belongs to.

4. **Locate Correct Sheet**
   - The function finds the sheet corresponding to that season.

5. **Find Conference Ranking**
   - Within that sheet, it searches for the row matching the team’s conference.

6. **Return Ranking**
   - If a match is found, the conference rank is returned.
   - If no match exists, the function returns a missing value (`NaN`).

The function is later applied to every row in the training dataset to populate a new **conference ranking feature**.

In [ ]:
def get_conf_rank(row, sheets):
    season = row["Season"]
    conf = row["Conference"]
    
    # Convert season to string in case it's numeric
    season = str(season)
    
    # Get the appropriate sheet
    if season in sheets:
        season_df = sheets[season]
        
        match = season_df.loc[
            season_df["Conference"] == conf,
            "Rank"
        ]
        
        if not match.empty:
            return match.iloc[0]
    
    return None  # if no match found

training["Conf Rank"] = np.nan
training["Conf Rank"] = training.apply(get_conf_rank, axis=1, args=(training_sheets,))

testing["Conf Rank"] = np.nan
testing["Conf Rank"] = training.apply(get_conf_rank, axis=1, args=(training_sheets,))



## Create Log-Transformed NET Ranking Feature

This cell creates a transformed version of the NET ranking statistic.

Steps performed:

1. **Calculate Logarithmic Transformation**
   - The natural logarithm of the `NET Rank` column is computed.

2. **Create a New Feature**
   - The transformed values are stored in a new column called `NET Rank Log`.

3. **Apply to Both Training and Testing Data**
   - The same transformation is applied to both datasets to maintain consistency.

Log transformations help machine learning models by:

- Reducing skew in ranking distributions
- Compressing large numeric ranges
- Improving model stability and performance

In [ ]:
training["NET Rank Log"] = np.log(training["NET Rank"])
testing["NET Rank Log"] = np.log(testing["NET Rank"])

ind_vars1 = ['AvgOppNETRank', 'AvgOppNET', 'Win', 'Loss', 'Conf W', 'Conf L', 'Non-Con W', 'Non-Con L', 'Road W', 'Road L', 'NETSOS', 'NETNonConfSOS', 'Q1 Win', 'Q1 Loss', 'Q2 Win', 'Q2 Loss', 'Q3 Win', 'Q3 Loss', 'Q4 Win', 'Q4 Loss', 'Conf Rank', "NET Rank Log"]

training_filt = training.dropna(subset = ind_vars1)
testing_filt = testing.dropna(subset = ind_vars1)




## Prepare Data for Seed Prediction Model

This cell prepares the dataset used to train the regression model that predicts NCAA tournament seeds.

Steps performed:

1. **Filter Training Data**
   - Only teams that received a tournament bid are included in this dataset.

2. **Select Predictor Variables**
   - `X_reg1` contains the features used to predict tournament seeding.

3. **Define Target Variable**
   - `y_reg` represents the actual tournament seed (`Overall Seed`).

4. **Prepare Model Inputs**
   - The model will learn how team statistics relate to seed placement.

These variables will be used to train the Gradient Boosting regression model in the next step.

In [ ]:
seed_train = training_filt.dropna(subset=["Bid Type"])
X_reg1 = seed_train[ind_vars1]
y_reg = seed_train["Overall Seed"]

XTrain, XTest, yTrain, yTest = train_test_split(
    X_reg1, y_reg, random_state=0, test_size=0.2
)

## Train Gradient Boosting Regression Model

This cell trains the machine learning model used to predict NCAA tournament seeds.

Model used:
**Gradient Boosting Regressor**

Gradient boosting works by:

1. Building multiple decision trees sequentially
2. Each tree correcting the errors of the previous trees
3. Combining the predictions of all trees to produce a final result

Model parameters include:

- **n_estimators = 2000**
  - The model builds 2000 decision trees.
- **learning_rate**
  - Controls how strongly each tree influences the final prediction.
- **max_depth**
  - Limits the depth of each decision tree to prevent overfitting.

This model learns patterns in historical data that relate team statistics to tournament seeding.

In [ ]:
# Gradient Boosting Model
gbr = GradientBoostingRegressor(
    n_estimators=2000,
    learning_rate=0.02,
    max_depth=2,
    subsample=0.7,
    validation_fraction=0.1,
    n_iter_no_change=25,
)

# Fit model
gbr.fit(XTrain, yTrain)

# Evaluate
train_pred = gbr.predict(XTrain)
yPred = gbr.predict(XTest)
print("Train R²:", round(metrics.r2_score(yTrain, train_pred), 3))
print("Test R²:", round(metrics.r2_score(yTest, yPred), 3))

# Create season groups for cross validation
groups = seed_train["Season"]

gkf = GroupKFold(n_splits=5)


# Cross-validation
scores = cross_val_score(gbr, X_reg1, y_reg, cv=gkf.split(X_reg1, y_reg, groups), scoring='r2')

print("Cross-validated R² scores:", scores)
print("Mean CV R²:", round(scores.mean(), 3))
print("Std Dev:", round(scores.std(), 3))

importance = pd.Series(
    gbr.feature_importances_,
    index=ind_vars1
).sort_values(ascending=False)

print("\nFeature Importances:")
print(importance)

Train R²: 0.97
Test R²: 0.931
Cross-validated R² scores: [0.94178001 0.94269737 0.91690132 0.90283882 0.93839039]
Mean CV R²: 0.929
Std Dev: 0.016

Feature Importances:
NET Rank Log     0.772791
Q1 Win           0.091155
NETSOS           0.023182
AvgOppNETRank    0.023119
Conf Rank        0.020246
AvgOppNET        0.019917
Loss             0.013755
Non-Con L        0.007727
Q2 Win           0.004419
Q4 Loss          0.003855
Q2 Loss          0.003430
Road L           0.003220
NETNonConfSOS    0.003063
Q1 Loss          0.002878
Win              0.001630
Q3 Win           0.001480
Non-Con W        0.001267
Q3 Loss          0.001214
Q4 Win           0.000711
Conf L           0.000450
Conf W           0.000274
Road W           0.000217
dtype: float64


## Generate Final Seed Predictions

This cell generates the final predicted seed values.

Steps performed:

1. **Create Prediction Dataset**
   - A copy of the testing dataset is created.

2. **Initialize Prediction Columns**
   - Columns for predicted seed values are created.

3. **Apply Regression Model**
   - The trained Gradient Boosting model predicts seed values based on team statistics.

4. **Rank Teams by Predicted Strength**
   - Teams are sorted according to their predicted seed score.

5. **Assign Final Seed Numbers**
   - Teams are ranked sequentially from best to worst.

The result is a dataset containing each team and its predicted NCAA tournament seed.

In [ ]:
final_pred = testing.copy()
final_pred["Initial Seed"] = np.nan

filtered_pred = testing_filt[
    (testing_filt["Bid Type"] == "AL") |
    (testing_filt["Bid Type"] == "AQ")
]

X_reg1_test = filtered_pred[ind_vars1]

predictions = gbr.predict(X_reg1_test)

final_pred.loc[filtered_pred.index, "Initial Seed"] = predictions

final_pred["Overall Seed"] = final_pred["Initial Seed"].round(0)
final_pred["Overall Seed"] = final_pred["Overall Seed"].fillna(0).astype(int)

final_pred = final_pred[["RecordID", "Overall Seed"]]

final_pred.to_csv("predictions.csv", index=False)

